In [1]:
import sys, os
path = os.path.abspath('../src')
sys.path.insert(0, path)

In [2]:
from gedih3.config import GH3_DEFAULT_DOWNLOAD_DIR, GH3_DEFAULT_SOC_DIR, GEDI_START_DATE
from gedih3.gedidriver import soc_file_tree, dask_h5_merged
from gedih3.gh3builder import h3_index_df

/home/tiago/miniconda3/envs/gedih3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from dask.distributed import Client
import dask.dataframe
import dask_geopandas
client = Client(processes=True, threads_per_worker=1, n_workers=2)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 2
Total threads: 2,Total memory: 15.49 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:44121,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:39477,Total threads: 1
Dashboard: http://127.0.0.1:42997/status,Memory: 7.74 GiB
Nanny: tcp://127.0.0.1:41467,


In [10]:
f = soc_file_tree(GH3_DEFAULT_SOC_DIR, to_list=True)
dat_col = 'delta_time'
lat_col = 'lat_lowestmode'
lon_col = 'lon_lowestmode'
ddf = dask_h5_merged(f, {'L2A':['rh','lon_lowestmode','lat_lowestmode','delta_time'], 'L4A': ['agbd'], 'L4C': ['wsci']}, by_beam=True)
ddf

,rh_000,rh_001,rh_002,rh_003,rh_004,rh_005,rh_006,rh_007,rh_008,rh_009,rh_010,rh_011,rh_012,rh_013,rh_014,rh_015,rh_016,rh_017,rh_018,rh_019,rh_020,rh_021,rh_022,rh_023,rh_024,rh_025,rh_026,rh_027,rh_028,rh_029,rh_030,rh_031,rh_032,rh_033,rh_034,rh_035,rh_036,rh_037,rh_038,rh_039,rh_040,rh_041,rh_042,rh_043,rh_044,rh_045,rh_046,rh_047,rh_048,rh_049,rh_050,rh_051,rh_052,rh_053,rh_054,rh_055,rh_056,rh_057,rh_058,rh_059,rh_060,rh_061,rh_062,rh_063,rh_064,rh_065,rh_066,rh_067,rh_068,rh_069,rh_070,rh_071,rh_072,rh_073,rh_074,rh_075,rh_076,rh_077,rh_078,rh_079,rh_080,rh_081,rh_082,rh_083,rh_084,rh_085,rh_086,rh_087,rh_088,rh_089,rh_090,rh_091,rh_092,rh_093,rh_094,rh_095,rh_096,rh_097,rh_098,rh_099,rh_100,lon_lowestmode,lat_lowestmode,delta_time,root_beam,root_file,agbd,wsci
npartitions=80,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float64,float64,float64,string,string,float32,float32
,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...


In [11]:
ddf['datetime'] = dask.dataframe.to_datetime(ddf[dat_col] + GEDI_START_DATE.timestamp(), unit='s')
ddf['year'] = ddf.datetime.dt.year

ddf = ddf.map_partitions(h3_index_df)
h3_tiles = ['838041fffffffff', '83804cfffffffff', '83804efffffffff']
ddf = ddf[ddf.h3_03.isin(h3_tiles)]

ddf = dask_geopandas.from_dask_dataframe(ddf, geometry=dask_geopandas.points_from_xy(ddf, x=lon_col, y=lat_col, crs='EPSG:4326'))
ddf

,shot_number,rh_000,rh_001,rh_002,rh_003,rh_004,rh_005,rh_006,rh_007,rh_008,rh_009,rh_010,rh_011,rh_012,rh_013,rh_014,rh_015,rh_016,rh_017,rh_018,rh_019,rh_020,rh_021,rh_022,rh_023,rh_024,rh_025,rh_026,rh_027,rh_028,rh_029,rh_030,rh_031,rh_032,rh_033,rh_034,rh_035,rh_036,rh_037,rh_038,rh_039,rh_040,rh_041,rh_042,rh_043,rh_044,rh_045,rh_046,rh_047,rh_048,rh_049,rh_050,rh_051,rh_052,rh_053,rh_054,rh_055,rh_056,rh_057,rh_058,rh_059,rh_060,rh_061,rh_062,rh_063,rh_064,rh_065,rh_066,rh_067,rh_068,rh_069,rh_070,rh_071,rh_072,rh_073,rh_074,rh_075,rh_076,rh_077,rh_078,rh_079,rh_080,rh_081,rh_082,rh_083,rh_084,rh_085,rh_086,rh_087,rh_088,rh_089,rh_090,rh_091,rh_092,rh_093,rh_094,rh_095,rh_096,rh_097,rh_098,rh_099,rh_100,lon_lowestmode,lat_lowestmode,delta_time,root_beam,root_file,agbd,wsci,datetime,year,h3_03,geometry
npartitions=80,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
,uint64,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float64,float64,float64,string,string,float32,float32,datetime64[ns],int32,object,geometry
,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...


In [12]:
ddf.head()

,shot_number,rh_000,rh_001,rh_002,rh_003,rh_004,rh_005,rh_006,rh_007,rh_008,...,lat_lowestmode,delta_time,root_beam,root_file,agbd,wsci,datetime,year,h3_03,geometry
h3_12,,,,,,,,,,,,,,,,,,,,,
8c8041b692493ff,61780000200098432,-2.35,-2.28,-2.17,-2.09,-1.98,-1.90,-1.83,-1.76,-1.72,...,0.194440,6.427411e+07,BEAM0000,GEDI02_A_2020014213209_O06178_02_T00931_02_003...,-9999.0,8.396494,2020-01-15 02:55:07.884629488,2020,838041fffffffff,POINT (-50.70178 0.19444)
8c8041b4da699ff,61780000200098492,-2.50,-2.39,-2.28,-2.20,-2.09,-2.02,-1.94,-1.87,-1.79,...,0.219602,6.427411e+07,BEAM0000,GEDI02_A_2020014213209_O06178_02_T00931_02_003...,-9999.0,8.294407,2020-01-15 02:55:08.380469561,2020,838041fffffffff,POINT (-50.68384 0.2196)
8c8041b4da6bbff,61780000200098493,-2.28,-2.20,-2.09,-2.02,-1.90,-1.83,-1.76,-1.72,-1.64,...,0.220020,6.427411e+07,BEAM0000,GEDI02_A_2020014213209_O06178_02_T00931_02_003...,-9999.0,8.246189,2020-01-15 02:55:08.388733625,2020,838041fffffffff,POINT (-50.68354 0.22002)
8c8041b4da4ddff,61780000200098494,-2.35,-2.28,-2.17,-2.09,-2.02,-1.94,-1.87,-1.79,-1.72,...,0.220440,6.427411e+07,BEAM0000,GEDI02_A_2020014213209_O06178_02_T00931_02_003...,-9999.0,8.085573,2020-01-15 02:55:08.396997452,2020,838041fffffffff,POINT (-50.68324 0.22044)
8c8041b4da481ff,61780000200098495,-2.39,-2.28,-2.17,-2.09,-1.98,-1.90,-1.83,-1.76,-1.68,...,0.220859,6.427411e+07,BEAM0000,GEDI02_A_2020014213209_O06178_02_T00931_02_003...,-9999.0,8.109316,2020-01-15 02:55:08.405261517,2020,838041fffffffff,POINT (-50.68294 0.22086)


In [7]:
ddf.to_parquet('/home/tiago/Desktop/repos/GEDI-H3/tmp/tmp/parquet_test', partition_on=['h3_03'])